In [1]:
import sys, os
sys.path.insert(0, 'src')
print(os.listdir('src'))

from pipeline import CompressorAnomalyPipeline
from explain import explain_alerts_batch
from train import engineer_features

import joblib
pipeline = joblib.load('models/compressor_pipeline_v3.pkl')

print(type(pipeline))
print("Feature columns:", pipeline.feature_cols)
print("Threshold:", pipeline.threshold_)

['pipeline.py', 'train.py', '__init__.py', '__pycache__', 'explain.py']
<class 'pipeline.CompressorAnomalyPipeline'>
Feature columns: ['H1_roll_std', 'H1_cycle_transitions', 'TP2_roll_std', 'TP2_roll_range', 'TP3_roll_std', 'TP3_roll_range', 'Reservoirs_roll_std', 'Reservoirs_roll_range', 'Oil_temperature_roll_mean', 'Oil_temperature_roll_trend', 'Motor_current_roll_std', 'Motor_current_roll_range']
Threshold: 0.027296743145617565


In [2]:
import pandas as pd

df = pd.read_csv('data/MetroPT3(AirCompressor).csv')
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

df = engineer_features(df)

fail4_chunk = df[(df['timestamp'] >= '2020-07-15 08:00') & (df['timestamp'] <= '2020-07-16 02:00')].copy()

alerts = pipeline.run(fail4_chunk)
alerts_with_language = explain_alerts_batch(alerts)

print(alerts['severity'].value_counts())
print("\nExample plain-language alert:")
print(alerts_with_language.iloc[0]['plain_language'])

Engineering features (this can take a few minutes on the full dataset)...
Feature engineering complete.
severity
INVESTIGATE    2456
ESCALATE       1443
MONITOR          34
Name: count, dtype: int64

Example plain-language alert:
This compressor is showing signs that are worth checking.

What we're seeing: oil temperature is running hotter than normal (2.1 standard deviations from normal), primary pressure sensor variability is more variable than usual (1.6 standard deviations from normal), and pressure cycling pattern is more irregular than usual (1.5 standard deviations from normal).

This pattern has been consistent for at least the last 10 of the last 10 minutes.

Recommended action: schedule an inspection in the near term.


In [3]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_rows = 90  # 15 minutes at 10s sampling — enough to fill the rolling window
start_time = pd.Timestamp('2026-01-01 00:00:00')
timestamps = [start_time + pd.Timedelta(seconds=10*i) for i in range(n_rows)]

# Simulate a compressor mid-failure: flat H1 (stuck running), elevated oil temp,
# reduced pressure swings — matching the real failure signature we found in EDA
fabricated = pd.DataFrame({
    'timestamp': timestamps,
    'TP2': np.random.normal(8.0, 0.3, n_rows),        # normally swings 0-10, here stuck mid-range
    'TP3': np.random.normal(8.0, 0.2, n_rows),
    'H1': np.random.normal(0.1, 0.05, n_rows),          # flat near zero = not cycling (failure signature)
    'DV_pressure': np.random.normal(0.0, 0.05, n_rows),
    'Reservoirs': np.random.normal(8.0, 0.3, n_rows),
    'Oil_temperature': np.random.normal(85.0, 1.5, n_rows),  # elevated vs normal ~55-65
    'Motor_current': np.random.normal(5.5, 0.3, n_rows),
})

print(fabricated.head())
print(fabricated.shape)

            timestamp       TP2       TP3        H1  DV_pressure  Reservoirs  \
0 2026-01-01 00:00:00  8.149014  8.019416  0.131283     0.072064    8.155804   
1 2026-01-01 00:00:10  7.958521  8.193729  0.057142    -0.071793    8.459822   
2 2026-01-01 00:00:20  8.194307  7.859589  0.046455     0.058158    7.967372   
3 2026-01-01 00:00:30  8.456909  7.934468  0.124124     0.000512    8.120514   
4 2026-01-01 00:00:40  7.929754  7.921578  0.088827    -0.049075    8.207043   

   Oil_temperature  Motor_current  
0        84.905981       5.750708  
1        86.432713       5.161088  
2        83.521411       5.658941  
3        85.756070       5.932471  
4        84.204614       4.758507  
(90, 8)


In [4]:
fabricated_features = engineer_features(fabricated.copy())

alerts = pipeline.run(fabricated_features)
alerts_with_language = explain_alerts_batch(alerts)

print(alerts[['timestamp', 'severity', 'flag_rate']])
if len(alerts_with_language) > 0:
    print("\nExample explanation:")
    print(alerts_with_language.iloc[-1]['plain_language'])
else:
    print("No alerts raised — check if fabricated pattern is anomalous enough")

Engineering features (this can take a few minutes on the full dataset)...
Feature engineering complete.
             timestamp  severity  flag_rate
0  2026-01-01 00:00:10  ESCALATE   0.500000
1  2026-01-01 00:00:20  ESCALATE   0.666667
2  2026-01-01 00:00:30  ESCALATE   0.750000
3  2026-01-01 00:00:40  ESCALATE   0.800000
4  2026-01-01 00:00:50  ESCALATE   0.833333
..                 ...       ...        ...
84 2026-01-01 00:14:10  ESCALATE   1.000000
85 2026-01-01 00:14:20  ESCALATE   1.000000
86 2026-01-01 00:14:30  ESCALATE   1.000000
87 2026-01-01 00:14:40  ESCALATE   1.000000
88 2026-01-01 00:14:50  ESCALATE   1.000000

[89 rows x 3 columns]

Example explanation:
URGENT: This compressor is showing strong signs of a developing problem.

What we're seeing: oil temperature is running hotter than normal (4.1 standard deviations from normal), motor current variability is unusually steady, which can indicate continuous running instead of normal cycling (1.1 standard deviations from norm

In [5]:
np.random.seed(7)

n_rows = 180  # 30 minutes at 10s sampling — enough to show a couple of real duty cycles
start_time = pd.Timestamp('2026-01-01 00:00:00')
timestamps = [start_time + pd.Timedelta(seconds=10*i) for i in range(n_rows)]

# Simulate NORMAL healthy duty-cycling: H1/TP2/Reservoirs sawtooth between low and high,
# oil temperature in its normal range, motor current cycling with the pressure
cycle_period = 20  # rows per on/off cycle
t = np.arange(n_rows)
sawtooth = 8 * ((t % cycle_period) / cycle_period)  # rises 0->8 then resets, repeat
is_on = (t % cycle_period) < (cycle_period * 0.7)   # "on" 70% of each cycle, "off" 30%

normal = pd.DataFrame({
    'timestamp': timestamps,
    'TP2': np.where(is_on, sawtooth + np.random.normal(0, 0.2, n_rows), np.random.normal(0.1, 0.05, n_rows)),
    'TP3': 8.0 + 0.5 * np.sin(t / 5) + np.random.normal(0, 0.15, n_rows),
    'H1': np.where(is_on, sawtooth + np.random.normal(0, 0.2, n_rows), np.random.normal(0.1, 0.05, n_rows)),
    'DV_pressure': np.random.normal(0.0, 0.05, n_rows),
    'Reservoirs': 8.0 + 0.5 * np.sin(t / 5) + np.random.normal(0, 0.15, n_rows),
    'Oil_temperature': 60.0 + 5 * np.sin(t / 30) + np.random.normal(0, 1.0, n_rows),  # normal range ~55-65
    'Motor_current': np.where(is_on, 5.5 + np.random.normal(0, 0.2, n_rows), np.random.normal(0.05, 0.02, n_rows)),
})

print(normal.head())
print(normal.shape)

            timestamp       TP2       TP3        H1  DV_pressure  Reservoirs  \
0 2026-01-01 00:00:00  0.338105  8.093994 -0.101461    -0.018387    7.826108   
1 2026-01-01 00:00:10  0.306813  8.250137  0.373752    -0.027174    7.824164   
2 2026-01-01 00:00:20  0.806564  8.121783  0.989170    -0.036371    8.007147   
3 2026-01-01 00:00:30  1.281503  8.316743  0.911114    -0.006845    8.134444   
4 2026-01-01 00:00:40  1.442215  8.260459  1.226485     0.039137    8.266993   

   Oil_temperature  Motor_current  
0        60.791927       4.963276  
1        61.925068       5.193041  
2        60.481006       5.585745  
3        61.056071       5.658424  
4        59.850669       5.239856  
(180, 8)


In [6]:
normal_features = engineer_features(normal.copy())

alerts_normal = pipeline.run(normal_features)

print(f"Alerts raised on normal data: {len(alerts_normal)}")
if len(alerts_normal) > 0:
    print(alerts_normal[['timestamp', 'severity', 'flag_rate']])
else:
    print("No alerts — pipeline correctly recognized this as normal operation")

Engineering features (this can take a few minutes on the full dataset)...
Feature engineering complete.
Alerts raised on normal data: 175
              timestamp  severity  flag_rate
0   2026-01-01 00:00:50   MONITOR   0.500000
1   2026-01-01 00:01:00   MONITOR   0.571429
2   2026-01-01 00:01:10   MONITOR   0.625000
3   2026-01-01 00:01:20   MONITOR   0.666667
4   2026-01-01 00:01:30   MONITOR   0.700000
..                  ...       ...        ...
170 2026-01-01 00:29:10  ESCALATE   1.000000
171 2026-01-01 00:29:20  ESCALATE   1.000000
172 2026-01-01 00:29:30  ESCALATE   1.000000
173 2026-01-01 00:29:40  ESCALATE   1.000000
174 2026-01-01 00:29:50  ESCALATE   1.000000

[175 rows x 3 columns]


In [7]:
alerts_normal_with_language = explain_alerts_batch(alerts_normal)
print(alerts_normal_with_language)  # should just be an empty dataframe, no error

              timestamp  severity  \
0   2026-01-01 00:00:50   MONITOR   
1   2026-01-01 00:01:00   MONITOR   
2   2026-01-01 00:01:10   MONITOR   
3   2026-01-01 00:01:20   MONITOR   
4   2026-01-01 00:01:30   MONITOR   
..                  ...       ...   
170 2026-01-01 00:29:10  ESCALATE   
171 2026-01-01 00:29:20  ESCALATE   
172 2026-01-01 00:29:30  ESCALATE   
173 2026-01-01 00:29:40  ESCALATE   
174 2026-01-01 00:29:50  ESCALATE   

                                          top_features  flag_rate  \
0    [(TP3_roll_range, 1.2598978188074828, 0.222748...   0.500000   
1    [(Oil_temperature_roll_trend, 1.33104452317587...   0.571429   
2    [(Oil_temperature_roll_trend, 1.74512871035032...   0.625000   
3    [(Motor_current_roll_std, 1.0881071342330542, ...   0.666667   
4    [(Oil_temperature_roll_trend, 1.44219521479696...   0.700000   
..                                                 ...        ...   
170  [(H1_cycle_transitions, 7.015414337602765, 8.0...   1.000000   
171

In [8]:
np.random.seed(7)

n_rows = 180
start_time = pd.Timestamp('2026-01-01 00:00:00')
timestamps = [start_time + pd.Timedelta(seconds=10*i) for i in range(n_rows)]

cycle_period = 60  # ~10 minutes per full on/off cycle — matches real data's ~1 transition/window
t = np.arange(n_rows)
sawtooth = 8 * ((t % cycle_period) / cycle_period)
is_on = (t % cycle_period) < (cycle_period * 0.7)

normal = pd.DataFrame({
    'timestamp': timestamps,
    'TP2': np.where(is_on, sawtooth + np.random.normal(0, 0.2, n_rows), np.random.normal(0.1, 0.05, n_rows)),
    'TP3': 8.0 + 0.5 * np.sin(t / 15) + np.random.normal(0, 0.15, n_rows),
    'H1': np.where(is_on, sawtooth + np.random.normal(0, 0.2, n_rows), np.random.normal(0.1, 0.05, n_rows)),
    'DV_pressure': np.random.normal(0.0, 0.05, n_rows),
    'Reservoirs': 8.0 + 0.5 * np.sin(t / 15) + np.random.normal(0, 0.15, n_rows),
    'Oil_temperature': 60.0 + 5 * np.sin(t / 60) + np.random.normal(0, 1.0, n_rows),
    'Motor_current': np.where(is_on, 5.5 + np.random.normal(0, 0.2, n_rows), np.random.normal(0.05, 0.02, n_rows)),
})

normal_features = engineer_features(normal.copy())
alerts_normal = pipeline.run(normal_features)
print(f"Alerts raised on normal data: {len(alerts_normal)}")
if len(alerts_normal) > 0:
    print(alerts_normal[['timestamp', 'severity', 'flag_rate']])

Engineering features (this can take a few minutes on the full dataset)...
Feature engineering complete.
Alerts raised on normal data: 172
              timestamp  severity  flag_rate
0   2026-01-01 00:00:10  ESCALATE   0.500000
1   2026-01-01 00:00:20  ESCALATE   0.666667
2   2026-01-01 00:00:30   MONITOR   0.500000
3   2026-01-01 00:01:50   MONITOR   0.500000
4   2026-01-01 00:02:00   MONITOR   0.538462
..                  ...       ...        ...
167 2026-01-01 00:29:10   MONITOR   0.950000
168 2026-01-01 00:29:20   MONITOR   0.950000
169 2026-01-01 00:29:30   MONITOR   0.950000
170 2026-01-01 00:29:40   MONITOR   0.950000
171 2026-01-01 00:29:50   MONITOR   0.950000

[172 rows x 3 columns]


In [9]:
alerts_normal_full = pipeline.run(normal_features)
for i in [0, 2, 50, 100, 171]:
    row = alerts_normal_full.iloc[i]
    print(row['timestamp'], row['severity'])
    print(row['top_features'])
    print()
    

2026-01-01 00:00:10 ESCALATE
[('Oil_temperature_roll_trend', np.float64(5.363819213693188), 0.5249172194384215, np.float64(-1.5859451828191003e-05)), ('Reservoirs_roll_range', np.float64(1.4575520597019938), 0.06796951880563107, np.float64(1.204113273232008)), ('TP3_roll_range', np.float64(1.4298608855353878), 0.09011652480621102, np.float64(1.2059220717828392))]

2026-01-01 00:00:30 MONITOR
[('TP3_roll_range', np.float64(1.301136047549368), 0.19056817692947892, np.float64(1.2059220717828392)), ('Reservoirs_roll_range', np.float64(1.296741177836176), 0.19331960689305205, np.float64(1.204113273232008)), ('Motor_current_roll_range', np.float64(1.1341698576447476), 0.6951484175001701, np.float64(3.6854442050003207))]

2026-01-01 00:09:40 MONITOR
[('Motor_current_roll_std', np.float64(1.2766628665230757), 2.5113109020209206, np.float64(1.2986011269403182)), ('H1_cycle_transitions', np.float64(1.1471867795230626), 2.0, np.float64(0.8270528691987564)), ('Motor_current_roll_range', np.float64

In [10]:
np.random.seed(7)

n_rows = 180
start_time = pd.Timestamp('2026-01-01 00:00:00')
timestamps = [start_time + pd.Timedelta(seconds=10*i) for i in range(n_rows)]

# Vary cycle length randomly around 60 rows (real compressors don't cycle on a perfect metronome)
t = np.arange(n_rows)
cycle_lengths = np.random.normal(60, 8, 10).astype(int).clip(40, 80)
cycle_bounds = np.cumsum(cycle_lengths)

is_on = np.zeros(n_rows, dtype=bool)
sawtooth = np.zeros(n_rows)
pos = 0
for cl in cycle_lengths:
    end = min(pos + cl, n_rows)
    on_len = int(cl * np.random.uniform(0.6, 0.8))
    seg = np.arange(end - pos)
    sawtooth[pos:end] = np.where(seg < on_len, 8 * (seg / max(on_len,1)), 0)
    is_on[pos:end] = seg < on_len
    pos = end
    if pos >= n_rows:
        break

normal = pd.DataFrame({
    'timestamp': timestamps,
    'TP2': np.where(is_on, sawtooth + np.random.normal(0, 0.3, n_rows), np.random.normal(0.1, 0.05, n_rows)),
    'TP3': 8.0 + 1.2 * np.sin(t / 15) + np.random.normal(0, 0.3, n_rows),   # bigger amplitude, matches real range ~1.2
    'H1': np.where(is_on, sawtooth + np.random.normal(0, 0.3, n_rows), np.random.normal(0.1, 0.05, n_rows)),
    'DV_pressure': np.random.normal(0.0, 0.1, n_rows),
    'Reservoirs': 8.0 + 1.2 * np.sin(t / 15) + np.random.normal(0, 0.3, n_rows),
    'Oil_temperature': 60.0 + 5 * np.sin(t / 60) + np.random.normal(0, 1.5, n_rows),
    'Motor_current': np.where(is_on, 5.5 + np.random.normal(0, 0.6, n_rows), np.random.normal(0.05, 0.05, n_rows)),  # more natural variability
})

normal_features = engineer_features(normal.copy())
alerts_normal = pipeline.run(normal_features)
print(f"Alerts raised on normal data: {len(alerts_normal)}")
if len(alerts_normal) > 0:
    print(alerts_normal[['timestamp','severity','flag_rate']])

Engineering features (this can take a few minutes on the full dataset)...
Feature engineering complete.
Alerts raised on normal data: 173
              timestamp     severity  flag_rate
0   2026-01-01 00:01:10  INVESTIGATE   0.500000
1   2026-01-01 00:01:20  INVESTIGATE   0.555556
2   2026-01-01 00:01:30  INVESTIGATE   0.600000
3   2026-01-01 00:01:40  INVESTIGATE   0.636364
4   2026-01-01 00:01:50  INVESTIGATE   0.666667
..                  ...          ...        ...
168 2026-01-01 00:29:10  INVESTIGATE   1.000000
169 2026-01-01 00:29:20  INVESTIGATE   1.000000
170 2026-01-01 00:29:30  INVESTIGATE   1.000000
171 2026-01-01 00:29:40  INVESTIGATE   1.000000
172 2026-01-01 00:29:50  INVESTIGATE   1.000000

[173 rows x 3 columns]


In [12]:
print(df.shape)
print(df['timestamp'].min(), df['timestamp'].max())

(1516948, 47)
2020-02-01 00:00:00 2020-09-01 03:59:50
